In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

train_raw = pd.read_csv('../data/train.csv', parse_dates=['Date'])
stores = pd.read_csv('../data/stores.csv')

store_weekly = (train_raw.groupby(['Store','Date'], as_index=False)['Weekly_Sales'].sum()
                .merge(stores[['Store','Type']], on='Store'))

dates = sorted(store_weekly['Date'].unique())
cutoff = dates[-13]
print('Cutoff:', cutoff)
print('Train weeks:', len(dates)-13, '| Test weeks:', 13)

Cutoff: 2012-08-03 00:00:00
Train weeks: 130 | Test weeks: 13


In [2]:
type_weekly = store_weekly.groupby(['Type','Date'], as_index=False)['Weekly_Sales'].sum()
total_weekly = store_weekly.groupby('Date', as_index=False)['Weekly_Sales'].sum()

def split(df):
    return df[df['Date'] < cutoff], df[df['Date'] >= cutoff]

store_tr, store_te = split(store_weekly)
type_tr,  type_te  = split(type_weekly)
total_tr, total_te = split(total_weekly)

print(total_tr.shape, total_te.shape)
print(type_tr.shape, type_te.shape)
print(store_tr.shape, store_te.shape)

(130, 2) (13, 2)
(390, 3) (39, 3)
(5850, 4) (585, 4)


In [3]:
def seasonal_naive(train_df, test_df, key=None):
    out = test_df.copy()
    lookup = train_df.set_index(([key] if key else []) + ['Date'])['Weekly_Sales']
    idx = out['Date'] - pd.Timedelta(weeks=52)
    if key:
        out['forecast'] = list(zip(out[key], idx))
    else:
        out['forecast'] = idx
    out['forecast'] = out['forecast'].map(lookup)
    return out

total_sn = seasonal_naive(total_tr, total_te)
print(total_sn[['Date','Weekly_Sales','forecast']].head())
print('Missing forecasts:', total_sn['forecast'].isna().sum())

          Date  Weekly_Sales     forecast
130 2012-08-03   47485899.56  48015466.97
131 2012-08-10   47403451.04  46249569.21
132 2012-08-17   47354452.05  46917347.62
133 2012-08-24   47447323.60  47416948.45
134 2012-08-31   47159639.43  45376623.27
Missing forecasts: 0


In [4]:
def mape(actual, forecast):
    return np.mean(np.abs((actual - forecast) / actual)) * 100

print('Total level MAPE:', round(mape(total_sn['Weekly_Sales'], total_sn['forecast']), 2), '%')

Total level MAPE: 2.04 %


In [5]:
type_sn  = seasonal_naive(type_tr,  type_te,  key='Type')
store_sn = seasonal_naive(store_tr, store_te, key='Store')

print('Missing —', 'type:', type_sn['forecast'].isna().sum(),
      '| store:', store_sn['forecast'].isna().sum())

print('\nMAPE by level')
print('Total:', round(mape(total_sn['Weekly_Sales'], total_sn['forecast']), 2), '%')
print('Type: ', round(mape(type_sn['Weekly_Sales'],  type_sn['forecast']),  2), '%')
print('Store:', round(mape(store_sn['Weekly_Sales'], store_sn['forecast']), 2), '%')

Missing — type: 0 | store: 0

MAPE by level
Total: 2.04 %
Type:  2.92 %
Store: 5.36 %


In [6]:
store_sum = store_sn.groupby('Date')['forecast'].sum().reset_index(name='from_stores')
cmp = total_sn[['Date','forecast']].merge(store_sum, on='Date')
cmp['gap'] = cmp['forecast'] - cmp['from_stores']

print(cmp[['Date','forecast','from_stores','gap']].to_string(index=False))
print('\nMax absolute gap:', round(cmp['gap'].abs().max(), 2))

      Date    forecast  from_stores  gap
2012-08-03 48015466.97  48015466.97  0.0
2012-08-10 46249569.21  46249569.21  0.0
2012-08-17 46917347.62  46917347.62  0.0
2012-08-24 47416948.45  47416948.45  0.0
2012-08-31 45376623.27  45376623.27  0.0
2012-09-07 46763227.53  46763227.53  0.0
2012-09-14 43793960.08  43793960.08  0.0
2012-09-21 42718096.73  42718096.73  0.0
2012-09-28 42195830.81  42195830.81  0.0
2012-10-05 47211688.36  47211688.36  0.0
2012-10-12 44374820.30  44374820.30  0.0
2012-10-19 45818953.44  45818953.44  0.0
2012-10-26 45855821.05  45855821.05  0.0

Max absolute gap: 0.0


In [7]:
baseline = {'total': total_sn, 'type': type_sn, 'store': store_sn}
for k, v in baseline.items():
    v.to_csv(f'../data/baseline_{k}.csv', index=False)
print('Saved.')

Saved.


Split by date, not randomly: last 13 weeks held out (cutoff 2012-08-03). Test window contains no Christmas, so errors understate holiday-period difficulty.

Seasonal naive MAPE: 2.04% total, 2.92% type, 5.36% store. Error doubles down the hierarchy as store-level noise stops cancelling. Baseline is coherent by construction (max gap 0.0).

In [8]:
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

y = total_tr.set_index('Date')['Weekly_Sales'].asfreq('W-FRI')
print(y.head())
print('Length:', len(y), '| Any NaN:', y.isna().sum())

Date
2010-02-05    49750740.50
2010-02-12    48336677.63
2010-02-19    48276993.78
2010-02-26    43968571.13
2010-03-05    46871470.30
Freq: W-FRI, Name: Weekly_Sales, dtype: float64
Length: 130 | Any NaN: 0


In [9]:
model = sm.tsa.statespace.SARIMAX(
    y,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 52),
    enforce_stationarity=False,
    enforce_invertibility=False
)
res = model.fit(disp=False)

fc = res.forecast(steps=13)
print(fc.head())
print('\nMAPE:', round(mape(total_te['Weekly_Sales'].values, fc.values), 2), '%')

2012-08-03    4.931507e+07
2012-08-10    4.755008e+07
2012-08-17    4.821721e+07
2012-08-24    4.870052e+07
2012-08-31    4.669291e+07
Freq: W-FRI, Name: predicted_mean, dtype: float64

MAPE: 1.87 %


In [10]:
def fit_forecast(series, steps=13):
    y = series.set_index('Date')['Weekly_Sales'].asfreq('W-FRI')
    m = sm.tsa.statespace.SARIMAX(y, order=(1,1,1), seasonal_order=(1,1,1,52),
                                  enforce_stationarity=False, enforce_invertibility=False)
    return m.fit(disp=False).forecast(steps=steps)

results = {}
results['total'] = fit_forecast(total_tr)
print('total done')

for t in ['A','B','C']:
    results[f'type_{t}'] = fit_forecast(type_tr[type_tr['Type']==t])
    print(f'type {t} done')

total done
type A done
type B done
type C done


In [11]:
import time
start = time.time()

for s in sorted(store_tr['Store'].unique()):
    results[f'store_{s}'] = fit_forecast(store_tr[store_tr['Store']==s])
    if s % 10 == 0:
        print(f'store {s} done, {round(time.time()-start)}s elapsed')

print(f'\nAll done. {len(results)} series fitted in {round(time.time()-start)}s')

store 10 done, 105s elapsed
store 20 done, 194s elapsed
store 30 done, 275s elapsed
store 40 done, 363s elapsed

All done. 49 series fitted in 412s


In [12]:
store_cols = [f'store_{s}' for s in sorted(store_tr['Store'].unique())]
type_cols  = ['type_A', 'type_B', 'type_C']

sum_stores = sum(results[c] for c in store_cols)
sum_types  = sum(results[c] for c in type_cols)
direct     = results['total']

gap_df = pd.DataFrame({
    'direct_total': direct,
    'sum_of_types': sum_types,
    'sum_of_stores': sum_stores
})
gap_df['gap_types']  = gap_df['direct_total'] - gap_df['sum_of_types']
gap_df['gap_stores'] = gap_df['direct_total'] - gap_df['sum_of_stores']

print(gap_df.round(0).to_string())
print('\nMax gap vs types: ', round(gap_df['gap_types'].abs().max(), 0))
print('Max gap vs stores:', round(gap_df['gap_stores'].abs().max(), 0))

            direct_total  sum_of_types  sum_of_stores  gap_types  gap_stores
2012-08-03    49315071.0    49269061.0     49053798.0    46010.0    261272.0
2012-08-10    47550078.0    47518399.0     47235048.0    31679.0    315030.0
2012-08-17    48217212.0    48214284.0     47947140.0     2929.0    270072.0
2012-08-24    48700524.0    48685179.0     47826741.0    15346.0    873783.0
2012-08-31    46692913.0    46654627.0     46829523.0    38285.0   -136610.0
2012-09-07    48050488.0    48008097.0     47446410.0    42392.0    604078.0
2012-09-14    45085065.0    45057496.0     44953566.0    27569.0    131499.0
2012-09-21    44002709.0    44001706.0     43720620.0     1002.0    282089.0
2012-09-28    43494284.0    43430541.0     43185414.0    63742.0    308869.0
2012-10-05    48489294.0    48491328.0     48105280.0    -2034.0    384014.0
2012-10-12    45660910.0    45643083.0     45392060.0    17827.0    268850.0
2012-10-19    47090066.0    47072270.0     46464374.0    17796.0    625692.0

In [13]:
fc_df = pd.DataFrame(results)
fc_df.to_csv('../data/base_forecasts.csv')
print(fc_df.shape)

(13, 49)


In [14]:
store_ids = sorted(store_tr['Store'].unique())
store_type = stores.set_index('Store')['Type'].to_dict()

rows = []
rows.append(np.ones(45))                                          # total
for t in ['A','B','C']:
    rows.append(np.array([1.0 if store_type[s]==t else 0.0 for s in store_ids]))
for i in range(45):
    r = np.zeros(45); r[i] = 1.0
    rows.append(r)

S = np.array(rows)
print('S shape:', S.shape)
print('Row sums:', S.sum(axis=1)[:4], '...')
print('Column sums:', S.sum(axis=0)[:5], '...')

S shape: (49, 45)
Row sums: [45. 22. 17.  6.] ...
Column sums: [3. 3. 3. 3. 3.] ...


In [15]:
P = S @ np.linalg.inv(S.T @ S) @ S.T

print('P shape:', P.shape)
print('Idempotent (P @ P == P):', np.allclose(P @ P, P))
print('Symmetric (P == P.T):   ', np.allclose(P, P.T))
print('Trace (should be 45):   ', round(np.trace(P), 6))

P shape: (49, 49)
Idempotent (P @ P == P): True
Symmetric (P == P.T):    True
Trace (should be 45):    45.0


In [16]:
order = ['total'] + type_cols + store_cols
Y = fc_df[order].values.T          # 49 series × 13 weeks

Y_rec = P @ Y

rec_df = pd.DataFrame(Y_rec.T, index=fc_df.index, columns=order)

chk = rec_df[store_cols].sum(axis=1)
print('Max gap after reconciliation:', round((rec_df['total'] - chk).abs().max(), 6))
print('Max gap before:              ', round(gap_df['gap_stores'].abs().max(), 0))

Max gap after reconciliation: 0.0
Max gap before:               873783.0


In [17]:
actual_total = total_te.set_index('Date')['Weekly_Sales']
actual_store = store_te.pivot(index='Date', columns='Store', values='Weekly_Sales')
actual_store.columns = [f'store_{c}' for c in actual_store.columns]

print('           before    after')
print('Total: ', round(mape(actual_total.values, fc_df['total'].values),2),
      '   ', round(mape(actual_total.values, rec_df['total'].values),2))
print('Store: ', round(mape(actual_store[store_cols].values.ravel(),
                            fc_df[store_cols].values.ravel()),2),
      '   ', round(mape(actual_store[store_cols].values.ravel(),
                        rec_df[store_cols].values.ravel()),2))

           before    after
Total:  1.87     1.86
Store:  3.79     4.19


In [18]:
rec_df.to_csv('../data/reconciled_ols.csv')
print('Saved.')

Saved.


In [19]:
def fit_resid(series):
    y = series.set_index('Date')['Weekly_Sales'].asfreq('W-FRI')
    m = sm.tsa.statespace.SARIMAX(y, order=(1,1,1), seasonal_order=(1,1,1,52),
                                  enforce_stationarity=False, enforce_invertibility=False)
    return m.fit(disp=False).resid

resids = {}
resids['total'] = fit_resid(total_tr)
for t in ['A','B','C']:
    resids[f'type_{t}'] = fit_resid(type_tr[type_tr['Type']==t])
print('4 done')

4 done


In [20]:
import time
start = time.time()

for s in store_ids:
    resids[f'store_{s}'] = fit_resid(store_tr[store_tr['Store']==s])
    if s % 10 == 0:
        print(f'store {s}, {round(time.time()-start)}s')

print(f'\n{len(resids)} residual series in {round(time.time()-start)}s')

store 10, 104s
store 20, 192s
store 30, 272s
store 40, 359s

49 residual series in 408s


In [21]:
R = pd.DataFrame(resids)[order]

burn = 52
R = R.iloc[burn:]
print('Residual matrix:', R.shape)

w = R.var().values
W_inv = np.diag(1.0 / w)

P_mint = S @ np.linalg.inv(S.T @ W_inv @ S) @ S.T @ W_inv

print('Idempotent:', np.allclose(P_mint @ P_mint, P_mint))
print('Symmetric: ', np.allclose(P_mint, P_mint.T))
print('Trace:     ', round(np.trace(P_mint), 6))

Residual matrix: (78, 49)
Idempotent: True
Symmetric:  False
Trace:      45.0


In [22]:
Y_mint = P_mint @ Y
mint_df = pd.DataFrame(Y_mint.T, index=fc_df.index, columns=order)

print('Coherence gap:', round((mint_df['total'] - mint_df[store_cols].sum(axis=1)).abs().max(), 6))

print('\n              base    OLS     MinT')
print('Total:', 
      round(mape(actual_total.values, fc_df['total'].values),2),
      round(mape(actual_total.values, rec_df['total'].values),2),
      round(mape(actual_total.values, mint_df['total'].values),2))
print('Store:',
      round(mape(actual_store[store_cols].values.ravel(), fc_df[store_cols].values.ravel()),2),
      round(mape(actual_store[store_cols].values.ravel(), rec_df[store_cols].values.ravel()),2),
      round(mape(actual_store[store_cols].values.ravel(), mint_df[store_cols].values.ravel()),2))

Coherence gap: 0.0

              base    OLS     MinT
Total: 1.87 1.86 1.57
Store: 3.79 4.19 3.8


In [23]:
mint_df.to_csv('../data/reconciled_mint.csv')

comparison = pd.DataFrame({
    'base': [mape(actual_total.values, fc_df['total'].values),
             mape(actual_store[store_cols].values.ravel(), fc_df[store_cols].values.ravel())],
    'ols':  [mape(actual_total.values, rec_df['total'].values),
             mape(actual_store[store_cols].values.ravel(), rec_df[store_cols].values.ravel())],
    'mint': [mape(actual_total.values, mint_df['total'].values),
             mape(actual_store[store_cols].values.ravel(), mint_df[store_cols].values.ravel())]
}, index=['total','store']).round(2)
comparison.to_csv('../data/reconciliation_comparison.csv')
print(comparison)

       base   ols  mint
total  1.87  1.86  1.57
store  3.79  4.19  3.80


In [24]:
def fit_with_se(series, steps=13):
    y = series.set_index('Date')['Weekly_Sales'].asfreq('W-FRI')
    m = sm.tsa.statespace.SARIMAX(y, order=(1,1,1), seasonal_order=(1,1,1,52),
                                  enforce_stationarity=False, enforce_invertibility=False)
    r = m.fit(disp=False).get_forecast(steps=steps)
    return r.predicted_mean, r.se_mean

mean_t, se_t = fit_with_se(total_tr)
print(pd.DataFrame({'mean': mean_t, 'se': se_t}).round(0).head())

                  mean         se
2012-08-03  49315071.0  2124066.0
2012-08-10  47550078.0  2129431.0
2012-08-17  48217212.0  2134655.0
2012-08-24  48700524.0  2139867.0
2012-08-31  46692913.0  2145066.0


In [25]:
from scipy.stats import norm

Cu, Co = 3.0, 1.0
fractile = Cu / (Cu + Co)

order_qty = mean_t + norm.ppf(fractile) * se_t

decision = pd.DataFrame({
    'forecast': mean_t,
    'se': se_t,
    'order': order_qty,
    'buffer': order_qty - mean_t
}).round(0)

print('Critical fractile:', fractile)
print('z-score:', round(norm.ppf(fractile), 4))
print()
print(decision.head())

Critical fractile: 0.75
z-score: 0.6745

              forecast         se       order     buffer
2012-08-03  49315071.0  2124066.0  50747732.0  1432661.0
2012-08-10  47550078.0  2129431.0  48986357.0  1436279.0
2012-08-17  48217212.0  2134655.0  49657015.0  1439803.0
2012-08-24  48700524.0  2139867.0  50143843.0  1443318.0
2012-08-31  46692913.0  2145066.0  48139737.0  1446825.0


In [26]:
scenarios = {'Fresh grocery (Cu=1, Co=3)': (1,3),
             'Balanced (Cu=1, Co=1)':      (1,1),
             'General merch (Cu=3, Co=1)': (3,1),
             'Critical stock (Cu=9, Co=1)': (9,1)}

rows = []
for name, (cu, co) in scenarios.items():
    f = cu/(cu+co)
    q = mean_t.iloc[0] + norm.ppf(f)*se_t.iloc[0]
    rows.append({'scenario': name, 'fractile': round(f,2),
                 'z': round(norm.ppf(f),3),
                 'order': round(q,0),
                 'vs_forecast': round(q - mean_t.iloc[0], 0)})

print(pd.DataFrame(rows).to_string(index=False))

                   scenario  fractile      z      order  vs_forecast
 Fresh grocery (Cu=1, Co=3)      0.25 -0.674 47882410.0   -1432661.0
      Balanced (Cu=1, Co=1)      0.50  0.000 49315071.0          0.0
 General merch (Cu=3, Co=1)      0.75  0.674 50747732.0    1432661.0
Critical stock (Cu=9, Co=1)      0.90  1.282 52037171.0    2722100.0


In [27]:
decision.to_csv('../data/order_decisions.csv')
pd.DataFrame(rows).to_csv('../data/newsvendor_scenarios.csv', index=False)
print('Saved.')

Saved.


SARIMA gives a mean and standard error per week; the newsvendor critical fractile Cu/(Cu+Co) selects which quantile of that distribution to order at.

At Cu=3, Co=1 the fractile is 0.75, so orders sit ~£1.43M above the forecast. The buffer widens with forecast horizon as se grows. Ordering the point forecast is optimal only when Cu=Co.

In [28]:
from scipy import stats

r_total = resids['total'].iloc[52:]

print('Skew:    ', round(stats.skew(r_total), 3))
print('Kurtosis:', round(stats.kurtosis(r_total), 3))

stat, p = stats.jarque_bera(r_total)
print('Jarque-Bera p:', round(p, 4))

Skew:     -1.614
Kurtosis: 1.973
Jarque-Bera p: 0.0


In [29]:
r = resids['total'].iloc[52:].values

emp_q = np.quantile(r, fractile)
nrm_q = norm.ppf(fractile) * r.std()

print('Empirical 75th pct of residuals:', round(emp_q, 0))
print('Normal-implied 75th pct:        ', round(nrm_q, 0))
print('Difference:                     ', round(emp_q - nrm_q, 0))

Empirical 75th pct of residuals: -153369.0
Normal-implied 75th pct:         5587107.0
Difference:                      -5740476.0


In [30]:
print('Mean residual:  ', round(r.mean(), 0))
print('Median residual:', round(np.median(r), 0))
print('Std residual:   ', round(r.std(), 0))
print()
print('Quantiles:')
for q in [0.05, 0.25, 0.5, 0.75, 0.95]:
    print(f'  {q}: {round(np.quantile(r, q), 0)}')

Mean residual:   -5472153.0
Median residual: -1876466.0
Std residual:    8283457.0

Quantiles:
  0.05: -23979272.0
  0.25: -7364116.0
  0.5: -1876466.0
  0.75: -153369.0
  0.95: 1295302.0


In [31]:
r_full = resids['total']
print('Full length:', len(r_full))
print()
for start in [0, 52, 60, 70, 80]:
    seg = r_full.iloc[start:]
    print(f'from {start:3d}: n={len(seg):3d}  mean={seg.mean():>12,.0f}  std={seg.std():>12,.0f}')

Full length: 130

from   0: n=130  mean=  -3,108,918  std=   9,354,825
from  52: n= 78  mean=  -5,472,153  std=   8,337,073
from  60: n= 70  mean=  -3,247,394  std=   5,100,738
from  70: n= 60  mean=  -1,575,357  std=   2,760,781
from  80: n= 50  mean=    -706,886  std=   2,043,999


In [32]:
BURN = 80
R2 = pd.DataFrame(resids)[order].iloc[BURN:]
print('Residual matrix:', R2.shape)

w2 = R2.var().values
W2_inv = np.diag(1.0 / w2)
P_mint2 = S @ np.linalg.inv(S.T @ W2_inv @ S) @ S.T @ W2_inv

print('Idempotent:', np.allclose(P_mint2 @ P_mint2, P_mint2))
print('Trace:     ', round(np.trace(P_mint2), 6))

mint2 = pd.DataFrame((P_mint2 @ Y).T, index=fc_df.index, columns=order)
print('Coherence gap:', round((mint2['total'] - mint2[store_cols].sum(axis=1)).abs().max(), 6))

print('\n            MinT(52)  MinT(80)')
print('Total:', round(mape(actual_total.values, mint_df['total'].values),2),
      '    ', round(mape(actual_total.values, mint2['total'].values),2))
print('Store:', round(mape(actual_store[store_cols].values.ravel(), mint_df[store_cols].values.ravel()),2),
      '    ', round(mape(actual_store[store_cols].values.ravel(), mint2[store_cols].values.ravel()),2))

Residual matrix: (50, 49)
Idempotent: True
Trace:      45.0
Coherence gap: 0.0

            MinT(52)  MinT(80)
Total: 1.57      1.63
Store: 3.8      3.77


In [33]:
r_clean = resids['total'].iloc[BURN:].values

print('Mean:  ', round(r_clean.mean(), 0))
print('Median:', round(np.median(r_clean), 0))
print('Skew:  ', round(stats.skew(r_clean), 3))
print('Kurt:  ', round(stats.kurtosis(r_clean), 3))
print('JB p:  ', round(stats.jarque_bera(r_clean)[1], 4))

Mean:   -706886.0
Median: -589257.0
Skew:   0.311
Kurt:   1.983
JB p:   0.0111


In [34]:
emp_q2 = np.quantile(r_clean, fractile)
nrm_q2 = norm.ppf(fractile) * r_clean.std()

print('Empirical 75th pct:', round(emp_q2, 0))
print('Normal-implied:    ', round(nrm_q2, 0))
print('Difference:        ', round(emp_q2 - nrm_q2, 0))
print()
print('Order (normal):   ', round(mean_t.iloc[0] + norm.ppf(fractile)*se_t.iloc[0], 0))
print('Order (empirical):', round(mean_t.iloc[0] + emp_q2, 0))

Empirical 75th pct: 380811.0
Normal-implied:     1364800.0
Difference:         -983990.0

Order (normal):    50747732.0
Order (empirical): 49695882.0


In [35]:
mint2.to_csv('../data/reconciled_mint_burn80.csv')

final = pd.DataFrame({
    'total': [mape(actual_total.values, fc_df['total'].values),
              mape(actual_total.values, rec_df['total'].values),
              mape(actual_total.values, mint2['total'].values)],
    'store': [mape(actual_store[store_cols].values.ravel(), fc_df[store_cols].values.ravel()),
              mape(actual_store[store_cols].values.ravel(), rec_df[store_cols].values.ravel()),
              mape(actual_store[store_cols].values.ravel(), mint2[store_cols].values.ravel())]
}, index=['base','ols','mint']).round(2)
final.to_csv('../data/final_comparison.csv')
print(final)

      total  store
base   1.87   3.79
ols    1.86   4.19
mint   1.63   3.77


In [36]:
def run_origin(cut_idx):
    cut = dates[cut_idx]
    s_tr, s_te = store_weekly[store_weekly['Date']<cut], store_weekly[(store_weekly['Date']>=cut)]
    s_te = s_te[s_te['Date'] < dates[cut_idx+13]] if cut_idx+13 < len(dates) else s_te
    
    ty_tr = s_tr.groupby(['Type','Date'], as_index=False)['Weekly_Sales'].sum()
    to_tr = s_tr.groupby('Date', as_index=False)['Weekly_Sales'].sum()
    to_te = s_te.groupby('Date', as_index=False)['Weekly_Sales'].sum()
    
    res, rsd = {}, {}
    res['total'], rsd['total'] = fit_forecast(to_tr), fit_resid(to_tr)
    for t in ['A','B','C']:
        sub = ty_tr[ty_tr['Type']==t]
        res[f'type_{t}'], rsd[f'type_{t}'] = fit_forecast(sub), fit_resid(sub)
    for s in store_ids:
        sub = s_tr[s_tr['Store']==s]
        res[f'store_{s}'], rsd[f'store_{s}'] = fit_forecast(sub), fit_resid(sub)
    
    F = pd.DataFrame(res)[order]
    Rm = pd.DataFrame(rsd)[order].iloc[BURN:]
    Wi = np.diag(1.0/Rm.var().values)
    Pm = S @ np.linalg.inv(S.T @ Wi @ S) @ S.T @ Wi
    M = pd.DataFrame((Pm @ F.values.T).T, index=F.index, columns=order)
    
    a = to_te['Weekly_Sales'].values[:13]
    return (mape(a, F['total'].values[:13]), mape(a, M['total'].values[:13]))

import time
origins = [96, 104, 112, 120, 130]
out = []
for c in origins:
    t0 = time.time()
    b, m = run_origin(c)
    out.append({'cutoff_week': c, 'base': round(b,2), 'mint': round(m,2)})
    print(f'origin {c}: base {b:.2f}  mint {m:.2f}  ({round(time.time()-t0)}s)')

cv = pd.DataFrame(out)
print()
print(cv.to_string(index=False))
print('\nMinT better in', (cv['mint'] < cv['base']).sum(), 'of', len(cv), 'windows')

origin 96: base 3.10  mint 3.14  (99s)
origin 104: base 3.63  mint 3.13  (104s)
origin 112: base 3.15  mint 2.87  (1298s)
origin 120: base 2.53  mint 2.28  (910s)
origin 130: base 1.87  mint 1.63  (521s)

 cutoff_week  base  mint
          96  3.10  3.14
         104  3.63  3.13
         112  3.15  2.87
         120  2.53  2.28
         130  1.87  1.63

MinT better in 4 of 5 windows


In [37]:
cv.to_csv('../data/rolling_origin_cv.csv', index=False)
print(cv.to_string(index=False))
print('\nMinT better in', (cv['mint'] < cv['base']).sum(), 'of', len(cv))
print('Mean improvement:', round((cv['base'] - cv['mint']).mean(), 3), 'pp')

 cutoff_week  base  mint
          96  3.10  3.14
         104  3.63  3.13
         112  3.15  2.87
         120  2.53  2.28
         130  1.87  1.63

MinT better in 4 of 5
Mean improvement: 0.246 pp


In [38]:
mean_mint = mint2['total']
se_mint = se_t.copy()
se_mint.index = mean_mint.index

order_base = mean_t + norm.ppf(fractile) * se_t
order_mint = mean_mint + norm.ppf(fractile) * se_mint

cmp_order = pd.DataFrame({
    'actual': actual_total.values,
    'fc_base': mean_t.values,
    'fc_mint': mean_mint.values,
    'order_base': order_base.values,
    'order_mint': order_mint.values
}, index=mean_mint.index)
cmp_order['order_diff'] = cmp_order['order_mint'] - cmp_order['order_base']

print(cmp_order.round(0).to_string())
print('\nMean absolute order difference:', round(cmp_order['order_diff'].abs().mean(), 0))

                actual     fc_base     fc_mint  order_base  order_mint  order_diff
2012-08-03  47485900.0  49315071.0  49102479.0  50747732.0  50535140.0   -212592.0
2012-08-10  47403451.0  47550078.0  47317002.0  48986357.0  48753281.0   -233076.0
2012-08-17  47354452.0  48217212.0  48015088.0  49657015.0  49454891.0   -202124.0
2012-08-24  47447324.0  48700524.0  48081413.0  50143843.0  49524731.0   -619111.0
2012-08-31  47159639.0  46692913.0  46769314.0  48139737.0  48216139.0     76402.0
2012-09-07  48330059.0  48050488.0  47616454.0  49500811.0  49066777.0   -434034.0
2012-09-14  44226039.0  45085065.0  44975351.0  46538877.0  46429164.0   -109714.0
2012-09-21  44354547.0  44002709.0  43792765.0  45460003.0  45250059.0   -209943.0
2012-09-28  43734899.0  43494284.0  43250427.0  44955051.0  44711194.0   -243856.0
2012-10-05  47566639.0  48489294.0  48217667.0  49953526.0  49681899.0   -271627.0
2012-10-12  46128514.0  45660910.0  45462292.0  47128599.0  46929980.0   -198618.0
2012

In [39]:
sample_stores = ['store_1', 'store_4', 'store_36']

resid_sd = pd.DataFrame(resids)[order].iloc[BURN:].std()

rows_s = []
for c in sample_stores:
    sid = int(c.split('_')[1])
    typ = store_type[sid]
    fc0 = mint2[c].iloc[0]
    sd0 = resid_sd[c]
    q = fc0 + norm.ppf(fractile) * sd0
    rows_s.append({'store': sid, 'type': typ,
                   'forecast': round(fc0, 0),
                   'resid_sd': round(sd0, 0),
                   'order': round(q, 0),
                   'buffer': round(q - fc0, 0),
                   'buffer_pct': round(100*(q-fc0)/fc0, 2)})

store_orders = pd.DataFrame(rows_s)
print(store_orders.to_string(index=False))

 store type  forecast  resid_sd     order  buffer  buffer_pct
     1    A 1714924.0   79319.0 1768424.0 53500.0        3.12
     4    A 2296541.0   74231.0 2346609.0 50068.0        2.18
    36    A  301111.0   37166.0  326179.0 25068.0        8.33


In [40]:
cmp_order.to_csv('../data/order_comparison.csv')
store_orders.to_csv('../data/store_orders.csv', index=False)
print('Saved.')

Saved.


Rolling-origin validation across 5 cutoffs: MinT beat independent forecasts in 4 of 5 windows, mean improvement 0.246pp. The single loss was at the shortest training window, where W is least reliably estimated.

Order decisions rebuilt on reconciled forecasts: mean absolute change £291K/week, negative in 12 of 13 weeks — unreconciled forecasts were systematically high, so reconciliation reduces over-ordering.

Store-level buffers range 2.18% (store 4) to 8.33% (store 36). Small stores carry proportionally more forecast noise and need proportionally more safety stock; a flat buffer rule would misallocate inventory in both directions.

## Limitations

**Test window excludes Christmas.** The 13 held-out weeks run August–October 2012. The annual peak — the hardest week to forecast — is never evaluated. Reported MAPEs understate holiday-period difficulty.

**SARIMA parameters not tuned.** All 49 series use (1,1,1)(1,1,1,52). Individual stores likely warrant different orders; 49 grid searches was outside scope. The comparison across methods is still valid since all three use identical base forecasts.

**Two and a half seasonal cycles.** 130 training weeks against a 52-week period. Thin for estimating annual seasonality, and the reason `enforce_stationarity` and `enforce_invertibility` were disabled.

**Residual burn-in was initially wrong.** W was first estimated from index 52 (the seasonal period). Diagnostics showed the Kalman filter had not converged: residual std was £8.3M at index 52 versus £2.0M at index 80, the latter matching SARIMA's reported forecast SE of £2.1M. Re-estimating from index 80 changed total MAPE from 1.57% to 1.63% — the MinT conclusion held, but the original figure was contaminated. All results above use burn-in 80.

**Residuals are not normal.** Skew 0.311, kurtosis 1.983, Jarque-Bera p = 0.011. The empirical 75th percentile of residuals (£380,811) is 3.6× smaller than the normal-implied value (£1,364,800), a £1.05M difference in weekly order quantity. Fat tails inflate the standard deviation, and the normal assumption spreads that inflation symmetrically. The empirical quantile is less biased but noisier — 50 observations is thin for estimating a tail quantile.

**Persistent negative residual mean.** −£707k on a ~£47M series, roughly 1.5% systematic over-prediction in sample. Not diagnosed further.

**Prediction intervals not reconciled.** Order quantities use standard errors from the unreconciled fit. Propagating the full covariance through the reconciliation is outside scope, so buffer widths are approximate even where point forecasts are reconciled.